 ### <font color="orange">🧠 GIAI ĐOẠN 3: KỶ NGUYÊN LLMs</font>
 #### Bài 28: [FINAL BOSS] Tự xây dựng cấu trúc GPT hoàn chỉnh
 
 Đây là bài tập tổng hợp tất cả kiến thức từ Bài 19 đến Bài 28.
 Em phải đóng vai một Core ML Engineer tại OpenAI để lắp ráp con GPT này.
 
 **Yêu cầu:** # 1. Không dùng gợi ý có sẵn.
 2. Tự xử lý logic truyền Mask qua các tầng.
 3. Tự hiện thực hóa công thức Positional Encoding:

   $ PE(pos, 2i) =   \sin(pos / 10000^{2i/d_{model}}) $

   $ PE(pos, 2i+1) = \cos(pos / 10000^{2i/d_{model}}) $



In [5]:
# ==========================================
# PHẦN 1: IMPORT CÁC THƯ VIỆN CẦN THIẾT
# ==========================================
import torch          # Công cụ cốt lõi để tính toán Tensor và làm Deep Learning
import torch.nn as nn # Module chứa các khối xây dựng mạng nơ-ron (Linear, Embedding...)
import math           # Dùng để tính các phép toán như Logarit cơ số e (cho Positional Encoding)

# ==========================================
# PHẦN 2: LỚP POSITIONAL ENCODING (MÃ HÓA VỊ TRÍ)
# Mục đích: Giúp Transformer biết từ nào đứng trước, từ nào đứng sau 
# (vì nó đọc tất cả các từ cùng một lúc chứ không đọc từ trái sang phải).
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        
        # 1. Tạo sẵn một ma trận toàn số 0.
        # - Số hàng là max_len (chiều dài tối đa của câu)
        # - Số cột là d_model (kích thước vector của mỗi từ)
        pe = torch.zeros(max_len, d_model)
        
        # 2. Tạo một cột số đếm từ 0 đến max_len - 1 (đại diện cho vị trí thứ 1, 2, 3... trong câu)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        
        # 3. Tính toán "bước sóng" bằng công thức toán học. 
        # Nhờ bước sóng khác nhau, các vector vị trí sẽ không bao giờ bị trùng lặp.
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        # 4. Áp dụng hàm Sin cho các cột chẵn (0, 2, 4...)
        pe[:, 0::2] = torch.sin(position * div_term) 
        # 5. Áp dụng hàm Cos cho các cột lẻ (1, 3, 5...)
        pe[:, 1::2] = torch.cos(position * div_term) 
        
        # 6. Lưu ma trận pe vào bộ nhớ của mô hình dưới dạng "buffer".
        # Dùng buffer thay vì Parameter vì ta KHÔNG MUỐN các số Sin/Cos này bị thay đổi khi huấn luyện.
        # unsqueeze(0) là để thêm chiều Batch vào đầu thành (1, max_len, d_model)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        # Cắt lấy ma trận pe vừa vặn với độ dài thực tế của câu x đang xét (x.size(1))
        # Sau đó cộng thẳng vào x. Xong bước này, x chứa cả ngữ nghĩa lẫn vị trí!
        return x + self.pe[:, :x.size(1)]

In [6]:
class GPTModel(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, dropout=0.1, max_len=100):
        super().__init__()
        
        # Lưu lại độ dài câu dài nhất mà mô hình này chịu đựng được
        self.max_len = max_len 
        
        # Cuốn "từ điển" biến ID (số nguyên) thành một vector (số thực)
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        
        # Bộ mã hóa vị trí đã định nghĩa ở trên
        self.pos_encoding = PositionalEncoding(d_model, max_len)
        
        # Cơ chế "nhắm mắt ngẫu nhiên" bỏ qua một số thông tin để chống học vẹt (Overfitting)
        self.dropout = nn.Dropout(dropout)
        
        # Tạo ra một danh sách chứa `num_layers` khối Transformer
        self.blocks = nn.ModuleList([
            # Khối não bộ chính! Chứa cơ chế Self-Attention để các từ chú ý lẫn nhau.
            nn.TransformerEncoderLayer(
                d_model=d_model, 
                nhead=num_heads, 
                dim_feedforward=d_model*4, 
                batch_first=True, # Nghĩa là chiều Batch đứng đầu (Batch, Seq, Feature)
                norm_first=True   # Chuẩn hóa trước để quá trình train ổn định hơn
            ) for _ in range(num_layers)
        ])
        
        # Lớp chuẩn hóa cuối cùng, giúp dữ liệu ổn định trước khi đưa ra dự đoán
        self.ln_f = nn.LayerNorm(d_model)
        
        # Biến vector đặc trưng (d_model) quay ngược trở lại thành điểm số của toàn bộ từ vựng (vocab_size)
        self.head = nn.Linear(d_model, vocab_size)

    # --- LUỒNG ĐI DỮ LIỆU KHI HUẤN LUYỆN (TRAINING) ---
    def forward(self, idx):
        # Lấy độ dài hiện tại của câu
        sz = idx.size(1) 
        device = idx.device
        
        # Quá trình: Nhúng từ -> Cộng vị trí -> Dropout
        x = self.token_embedding(idx) 
        x = self.pos_encoding(x)
        x = self.dropout(x)
        
        # --- TẠO CAUSAL MASK (MẶT NẠ TIÊN TRI) ---
        # 1. Tạo một ma trận tam giác dưới toàn số 1.
        # Những số 1 này đại diện cho quá khứ và hiện tại (vùng an toàn, được phép nhìn)
        mask_ones = torch.tril(torch.ones(sz, sz, device=device))
        
        # 2. Đổ giá trị âm vô cùng (-inf) vào những vùng tương lai (chỗ có số 0)
        # Điều này bắt mô hình phải tự đoán chữ tiếp theo thay vì "nhìn trộm" đáp án phía sau.
        mask = torch.zeros(sz, sz, device=device)
        mask = mask.masked_fill(mask_ones == 0, float('-inf'))
        
        # Cho dữ liệu chạy qua từng khối Transformer, nhớ truyền cái 'mask' bịt mắt vào
        for block in self.blocks:
            x = block(x, src_mask=mask)
        
        # Đi qua chuẩn hóa và lớp Linear để xuất ra điểm số dự báo (Logits)
        x = self.ln_f(x)
        logits = self.head(x)
        return logits

    # ==========================================
    # PHẦN 4: VÒNG LẶP SINH VĂN BẢN (ĐÃ NÂNG CẤP)
    # Thêm tính năng Temperature và Sampling để AI sáng tạo hơn
    # ==========================================
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        # temperature = 1.0: Bình thường
        # temperature < 1.0: An toàn, ít rủi ro, ít sai nhưng nhàm chán
        # temperature > 1.0: Sáng tạo, bay bổng, rủi ro nói nhảm cao
        
        for _ in range(max_new_tokens):
            # 1. Cắt gọt câu mồi nếu quá dài
            idx_cond = idx[:, -self.max_len:]
            
            # 2. Đưa vào mô hình lấy điểm số (Logits)
            logits = self(idx_cond)
            
            # 3. Chỉ lấy điểm số của từ cuối cùng
            logits_last_step = logits[:, -1, :] 
            
            # 4. ÁP DỤNG NHIỆT ĐỘ (TEMPERATURE)
            # Chia điểm số cho nhiệt độ. Nếu nhiệt độ nhỏ, điểm số sẽ bị phóng đại lên (tự tin hơn).
            logits_last_step = logits_last_step / temperature
            
            # 5. Chuyển thành xác suất (Probabilities)
            probs = torch.softmax(logits_last_step, dim=-1)
            
            # 6. LẤY MẪU NGẪU NHIÊN (SAMPLING) THAY VÌ TÌM KIẾM THAM LAM (ARGMAX)
            # torch.multinomial giống như quay vòng quay may mắn dựa trên xác suất probs
            idx_next = torch.multinomial(probs, num_samples=1) # Lấy 1 từ ngẫu nhiên dựa theo trọng số
            
            # 7. Nối từ mới vào câu mồi
            idx = torch.cat((idx, idx_next), dim=1)
            
        return idx

In [7]:
# ==========================================
# PHẦN 5: HÀM LẤY DỮ LIỆU HUẤN LUYỆN (BATCHING)
# ==========================================
def get_batch(data, seq_len, batch_size, device):
    # Chọn ngẫu nhiên 'batch_size' vị trí bắt đầu trong đoạn text
    ix = torch.randint(len(data) - seq_len, (batch_size,))
    
    # X là CÂU HỎI (từ vị trí i đến i + seq_len)
    # Y là ĐÁP ÁN (chính là X nhưng dịch sang phải 1 ký tự)
    # Ví dụ: Câu gốc là "Xin chào"
    # X = "Xin chà"
    # Y = "in chào" (Bắt mô hình nhìn chữ X phải đoán ra chữ i, nhìn chữ i phải đoán ra chữ n...)
    
    x = torch.stack([data[i:i+seq_len] for i in ix])
    y = torch.stack([data[i+1:i+seq_len+1] for i in ix])
    
    return x.to(device), y.to(device)



In [8]:
# ==========================================
# PHẦN 6: LÒ LUYỆN ĐAN (TRAINING LOOP) & CHẠY THỬ
# ==========================================
def main():
    print("🚀 BẮT ĐẦU XÂY DỰNG NHÀ MÁY VÀ ĐÀO TẠO CÔNG NHÂN...")
    
    # 1. TẠO NGƯỜI PHIÊN DỊCH VÀ DỮ LIỆU (TOKENIZER)
    # Thầy cho một câu dài hơn chút để mô hình có cái mà học
    data_mau = "Xin chào thầy Gemini, hôm nay trò học AI rất vui! Trò sẽ cố gắng để tự tay code được một cái ChatGPT mini."
    chars = sorted(list(set(data_mau))) 
    vocab_size = len(chars)
    
    stoi = { ch:i for i,ch in enumerate(chars) } 
    itos = { i:ch for i,ch in enumerate(chars) } 
    encode = lambda s: [stoi[c] for c in s if c in stoi] 
    decode = lambda l: ''.join([itos[i] for i in l])     
    
    # Chuyển toàn bộ dữ liệu văn bản thành một Tensor dài chứa các ID
    data_tensor = torch.tensor(encode(data_mau), dtype=torch.long)
    
    # 2. KHỞI TẠO MÔ HÌNH
    # Chuyển mô hình lên GPU nếu máy trò có card rời (như NVIDIA), không thì chạy CPU vẫn ok
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    d_model = 64 
    num_heads = 4
    num_layers = 4 
    max_len = 16 # Mô hình luyện tập nhìn 16 ký tự cùng lúc
    
    model = GPTModel(vocab_size, d_model, num_heads, num_layers, max_len=max_len)
    model = model.to(device)
    
    # 3. THIẾT LẬP LÒ LUYỆN ĐAN
    # Optimizer: Đóng vai trò là "Trưởng phòng nhân sự", đi tinh chỉnh lại công nhân (trọng số)
    # lr=1e-3 là Learning Rate (tốc độ học), lớn quá thì ngợp, nhỏ quá thì học chậm
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    
    # Loss function: Đóng vai trò "Giám đốc", dùng CrossEntropy để chấm điểm độ sai lệch
    criterion = nn.CrossEntropyLoss()
    
    batch_size = 8
    epochs = 500 # Bắt mô hình làm đi làm lại 500 vòng
    
    print(f"\n🔥 ĐANG ĐƯA VÀO LÒ LUYỆN {epochs} VÒNG (Chạy trên {device.upper()})...")
    model.train() # Bật chế độ "Đi học" (kích hoạt Dropout)
    
    # --- VÒNG LẶP HUẤN LUYỆN CỐT LÕI (TRAINING LOOP) ---
    for step in range(epochs):
        # 3.1 Bốc 1 xấp bài kiểm tra (X) và đáp án chuẩn (Y)
        X_batch, Y_batch = get_batch(data_tensor, max_len, batch_size, device)
        
        # 3.2 Xóa sạch trí nhớ (đạo hàm) của lần kiểm tra trước để làm bài mới
        optimizer.zero_grad()
        
        # 3.3 Học sinh bắt đầu làm bài (Đưa X vào mô hình lấy điểm số Logits)
        logits = model(X_batch) 
        
        # 3.4 Giám đốc chấm điểm (Tính Loss)
        # PyTorch bắt buộc phải gộp chiều Batch và Seq_len lại thành một hàng dài mới chấm điểm được
        B, T, C = logits.shape
        logits_reshaped = logits.view(B*T, C)
        targets_reshaped = Y_batch.view(B*T)
        
        loss = criterion(logits_reshaped, targets_reshaped)
        
        # 3.5 Lan truyền ngược: Phát "sổ tay rút kinh nghiệm" xuống cho các tầng mạng Nơ-ron
        loss.backward()
        
        # 3.6 Cập nhật kiến thức: Các công nhân tự điều chỉnh lại máy móc (trọng số)
        optimizer.step()
        
        # In ra màn hình mỗi 100 vòng để xem nó học hành tiến bộ (Loss giảm) không
        if step % 100 == 0:
            print(f"   ⏳ Vòng {step}/{epochs} - Sai số (Loss): {loss.item():.4f}")
            
    print("✅ ĐÀO TẠO HOÀN TẤT! CÔNG NHÂN ĐÃ LÀNH NGHỀ.")
    
    # ==========================================
    # 4. CHO GPT TỰ NÓI SAU KHI TỐT NGHIỆP
    # ==========================================
    model.eval() # Bật chế độ "Đi làm thực tế" (Tắt Dropout, tắt tính Gradient)
    
    cau_moi = "Xin chào" # Trò thử đổi chữ khác xem, ví dụ "Trò học"
    print(f"\n📝 Câu mồi ban đầu: '{cau_moi}'")
    
    # Mã hóa câu mồi
    input_ids = encode(cau_moi)
    start_context = torch.tensor([input_ids], dtype=torch.long, device=device) 
    
    # Ép nó nhả ra 30 ký tự tiếp theo
    generated_ids = model.generate(start_context, max_new_tokens=30)
    output_text = decode(generated_ids[0].tolist())
    
    print(f"✨ KẾT QUẢ GPT SAU KHI HỌC: '{output_text}'")

if __name__ == "__main__":
    main()

🚀 BẮT ĐẦU XÂY DỰNG NHÀ MÁY VÀ ĐÀO TẠO CÔNG NHÂN...

🔥 ĐANG ĐƯA VÀO LÒ LUYỆN 1000 VÒNG (Chạy trên CPU)...
   ⏳ Vòng 0/1000 - Sai số (Loss): 3.8194
   ⏳ Vòng 100/1000 - Sai số (Loss): 0.5756
   ⏳ Vòng 200/1000 - Sai số (Loss): 0.2481
   ⏳ Vòng 300/1000 - Sai số (Loss): 0.2066
   ⏳ Vòng 400/1000 - Sai số (Loss): 0.1625
   ⏳ Vòng 500/1000 - Sai số (Loss): 0.1704
   ⏳ Vòng 600/1000 - Sai số (Loss): 0.1043
   ⏳ Vòng 700/1000 - Sai số (Loss): 0.1284
   ⏳ Vòng 800/1000 - Sai số (Loss): 0.1153
   ⏳ Vòng 900/1000 - Sai số (Loss): 0.1303
✅ ĐÀO TẠO HOÀN TẤT! CÔNG NHÂN ĐÃ LÀNH NGHỀ.

📝 Câu mồi ban đầu: 'Xin chào'
✨ KẾT QUẢ GPT SAU KHI HỌC: 'Xin chào thầy Gemini, hôm nay trò học '
